## Example 1: Mint a short-term Bedrock API key

In [ ]:
import base64
import importlib.metadata
from urllib.parse import parse_qs, urlsplit

from botocore.credentials import Credentials

from aws_bedrock_token_generator import BedrockTokenGenerator

creds = Credentials("AKIDEXAMPLE", "wJalrXUtnFEMI/K7MDENG+bPxRfiCYEXAMPLEKEY")
decode = lambda token: base64.b64decode(token.removeprefix("bedrock-api-key-")).decode()

# Example 1: the token is "bedrock-api-key-" + base64 of a presigned bedrock.amazonaws.com URL
assert importlib.metadata.version("aws-bedrock-token-generator") == "1.1.0"
token = BedrockTokenGenerator().get_token(creds, "us-west-2")
url = decode(token)
query = parse_qs(urlsplit("https://" + url).query)
assert url.startswith("bedrock.amazonaws.com/?Action=CallWithBearerToken") and url.endswith("&Version=1"), url
assert query["X-Amz-Algorithm"] == ["AWS4-HMAC-SHA256"] and "/us-west-2/bedrock/aws4_request" in query["X-Amz-Credential"][0]
print("Example 1 OK:", token[:40] + "...")

## Example 2: Choose the lifetime (at most 12 hours)

In [ ]:
import base64
from datetime import timedelta
from urllib.parse import parse_qs, urlsplit

from botocore.credentials import Credentials

from aws_bedrock_token_generator import provide_token

creds = Credentials("AKIDEXAMPLE", "wJalrXUtnFEMI/K7MDENG+bPxRfiCYEXAMPLEKEY")
decode = lambda token: base64.b64decode(token.removeprefix("bedrock-api-key-")).decode()

class Static:  # a botocore-style credential provider: load() returns credentials
    def load(self):
        return creds

# Example 2: the expiry becomes X-Amz-Expires; zero or more than 12 hours is refused
token = provide_token(region="eu-central-1", aws_credentials_provider=Static(), expiry=timedelta(minutes=30))
assert parse_qs(urlsplit("https://" + decode(token)).query)["X-Amz-Expires"] == ["1800"]
for bad in (timedelta(0), timedelta(hours=13)):
    try:
        provide_token(region="eu-central-1", aws_credentials_provider=Static(), expiry=bad)
        raise AssertionError(bad)
    except ValueError:
        pass
print("Example 2 OK")

## Example 3: Temporary credentials carry their session token

## Example 4: The default credential chain and region from the environment

In [ ]:
import base64
import os

from aws_bedrock_token_generator import provide_token

# Example 4: with no provider, botocore's default chain is used; with no region, AWS_REGION
saved = {k: os.environ.get(k) for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_REGION", "AWS_SESSION_TOKEN")}
os.environ.update(AWS_ACCESS_KEY_ID="AKIDENV", AWS_SECRET_ACCESS_KEY="envsecret", AWS_REGION="ap-southeast-2")
os.environ.pop("AWS_SESSION_TOKEN", None)
try:
    url = base64.b64decode(provide_token().removeprefix("bedrock-api-key-")).decode()
    assert "AKIDENV%2F" in url and "%2Fap-southeast-2%2Fbedrock%2F" in url, url
finally:
    for k, v in saved.items():
        if v is None:
            os.environ.pop(k, None)
        else:
            os.environ[k] = v
print("Example 4 OK")

## Example 5: A missing region is an error, not a guess

In [ ]:
import os

from botocore.credentials import Credentials

from aws_bedrock_token_generator import BedrockTokenGenerator, provide_token

# Example 5: both entry points refuse to sign without a region
saved = os.environ.pop("AWS_REGION", None)
try:
    for call in (lambda: provide_token(), lambda: BedrockTokenGenerator().get_token(Credentials("a", "b"), "")):
        try:
            call()
            raise AssertionError("signed without a region")
        except ValueError as error:
            assert "region" in str(error).lower(), error
finally:
    if saved is not None:
        os.environ["AWS_REGION"] = saved
print("Example 5 OK")